In [2]:

!pip install transformers datasets evaluate iterative-stratification -q


In [3]:
import os
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_value_
from torch.utils.data import DataLoader

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoConfig,
    TrainingArguments, Trainer, set_seed
)
import evaluate
from sklearn.metrics import f1_score, hamming_loss, precision_score, recall_score

In [4]:
SEED = 42
set_seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [5]:
# Determine the storage location based on the execution environment
# If running on Google Colab, use Google Drive as storage
if 'google.colab' in str(get_ipython()):
    from google.colab import drive  # Import Google Drive mounting utility
    drive.mount('/content/drive')  # Mount Google Drive

    # Set base folder path for storing data on Google Drive
    base_folder= Path('/content/drive/MyDrive/data')

Mounted at /content/drive


In [6]:
TRAIN_PATH = "/content/drive/My Drive/emotion-detection-fall-2025-nlp/train.csv"
TEST_PATH  = "/content/drive/My Drive/emotion-detection-fall-2025-nlp/test.csv"


train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
train_df.head()

Train shape: (7724, 13)


,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2017-21441,“Worry is a down payment on a problem you may ...,0,1,0,0,0,0,1,0,0,0,1
1,2017-31535,Whatever you decide to do make sure it makes y...,0,0,0,0,1,1,1,0,0,0,0
2,2017-21068,@Max_Kellerman it also helps that the majorit...,1,0,1,0,1,0,1,0,0,0,0
3,2017-31436,Accept the challenges so that you can literall...,0,0,0,0,1,0,1,0,0,0,0
4,2017-22195,My roommate: it's okay that we can't spell bec...,1,0,1,0,0,0,0,0,0,0,0


In [7]:
import re
from html import unescape

def clean_tweet(text):
    if not isinstance(text, str):
        return ""
    text = unescape(text)
    text = text.replace("\n", " ")
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = text.replace('â€œ', '"').replace('â€™', "'").replace('ðŸ', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

TEXT_COL = "Tweet"
ID_COL = "ID"

train_df[TEXT_COL] = train_df[TEXT_COL].astype(str).apply(clean_tweet).str.lower()
test_df[TEXT_COL]  = test_df[TEXT_COL].astype(str).apply(clean_tweet).str.lower()

In [8]:
label_cols = [c for c in train_df.columns if c not in (TEXT_COL, ID_COL)]
print("Labels:", label_cols)


from sklearn.model_selection import train_test_split

train_plus_val_texts, test_internal_texts, train_plus_val_labels, test_internal_labels = train_test_split(
    train_df[TEXT_COL].values,
    train_df[label_cols].values,
    test_size=0.15, random_state=SEED, shuffle=True
)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_plus_val_texts, train_plus_val_labels,
    test_size=0.1765, random_state=SEED, shuffle=True
)

print("Train / Val / Internal-test sizes:", len(train_texts), len(val_texts), len(test_internal_texts))

Labels: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']
Train / Val / Internal-test sizes: 5406 1159 1159


In [9]:
def build_hf_dataset(texts, labels):
    return Dataset.from_dict({"text": list(texts), "labels": [list(map(int, row)) for row in labels]})

train_hf = build_hf_dataset(train_texts, train_labels)
val_hf   = build_hf_dataset(val_texts, val_labels)
test_internal_hf = build_hf_dataset(test_internal_texts, test_internal_labels)

dataset_dict = DatasetDict({"train": train_hf, "valid": val_hf})
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 5406
    })
    valid: Dataset({
        features: ['text', 'labels'],
        num_rows: 1159
    })
})

In [10]:
checkpoint = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)

MAX_LENGTH = 128

def hf_tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

dataset_dict = dataset_dict.map(hf_tokenize, batched=True, remove_columns=["text"])
test_internal_tokenized = test_internal_hf.map(hf_tokenize, batched=True, remove_columns=["text"])

dataset_dict.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_internal_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/5406 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

In [11]:
def compute_pos_weights(hf_train_dataset):
    all_labels = np.stack(hf_train_dataset["labels"])
    positives = all_labels.sum(axis=0).astype(float)
    negatives = all_labels.shape[0] - positives
    positives_safe = np.where(positives == 0, 1.0, positives)
    pos_weight = negatives / positives_safe
    return torch.tensor(pos_weight, dtype=torch.float)

pos_weights = compute_pos_weights(dataset_dict["train"])
print("pos_weights:", pos_weights.numpy())

pos_weights: [ 1.6962594  6.057441   1.6204556  4.7327676  1.6828784  8.241026
  2.397863   7.733441   2.366127  18.586956  19.633587 ]


In [12]:
num_labels = len(label_cols)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

config = AutoConfig.from_pretrained(checkpoint)
config.id2label = {i: n for i, n in enumerate(label_cols)}
config.label2id = {n: i for i, n in enumerate(label_cols)}
model.config = config

model.to(DEVICE)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      

In [13]:
f1_metric = evaluate.load("f1", module_type="metric")

def sigmoid_np(x):
    return 1/(1+np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid_np(logits)
    preds = (probs > 0.5).astype(int)
    labels = labels.astype(int)
    # macro F1 via sklearn for stability
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    ham_loss = hamming_loss(labels, preds)
    return {"f1_macro": float(f1_macro), "hamming_loss": float(ham_loss)}

In [14]:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float().to(model.device)
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss



In [15]:

output_dir = "/content/modernbert_multilabel"
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    seed=SEED,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    max_grad_norm=1.0
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["valid"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-4184722196.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  trainer = CustomTrainer(


In [16]:

train_start = time.time()
trainer.train()
train_time = time.time() - train_start
print(f"Training finished in {train_time/60:.2f} minutes")
print("Best checkpoint:", trainer.state.best_model_checkpoint)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jahnavi-13489 (jahnavi-13489-1) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W1117 01:16:01.442000 3462 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Step,Training Loss,Validation Loss,F1 Macro,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
200,1.023400,0.960077,0.447538,0.314613,6.419800,180.535000,11.371000
400,0.818700,0.865720,0.492881,0.265040,3.508800,330.310000,20.805000
600,0.814300,0.789703,0.547847,0.225508,3.654800,317.121000,19.974000
800,0.610400,0.857809,0.561429,0.200800,3.577600,323.959000,20.405000
1000,0.651500,0.798407,0.548179,0.211389,3.607800,321.251000,20.234000
1200,0.639200,0.779003,0.562597,0.201114,3.591800,322.683000,20.324000
1400,0.456500,0.943972,0.574847,0.174210,3.589700,322.868000,20.336000
1600,0.447200,1.080615,0.580364,0.162679,3.606400,321.376000,20.242000
1800,0.449700,0.979669,0.570394,0.177347,3.584400,323.346000,20.366000
2000,0.473800,1.014601,0.578349,0.161660,3.592600,322.606000,20.319000


Training finished in 9.74 minutes
Best checkpoint: /content/modernbert_multilabel/checkpoint-1600


In [17]:

eval_results = trainer.evaluate(dataset_dict["valid"])
print("Validation results:", eval_results)


test_results = trainer.predict(test_internal_tokenized)
test_logits = test_results.predictions
test_probs = sigmoid_np(test_logits)
test_labels_arr = np.array(test_results.label_ids).astype(int)


test_preds = (test_probs >= 0.5).astype(int)
test_f1_macro = f1_score(test_labels_arr, test_preds, average="macro", zero_division=0)
test_hamming = hamming_loss(test_labels_arr, test_preds)
print("Internal test f1_macro (0.5):", test_f1_macro, "hamming:", test_hamming)


Validation results: {'eval_loss': 1.0806150436401367, 'eval_f1_macro': 0.5803636905102892, 'eval_hamming_loss': 0.16267942583732056, 'eval_runtime': 3.6249, 'eval_samples_per_second': 319.733, 'eval_steps_per_second': 20.138, 'epoch': 5.0}
Internal test f1_macro (0.5): 0.5812543107688306 hamming: 0.16628755196485998


In [18]:

valid_out = trainer.predict(dataset_dict["valid"])
valid_logits = valid_out.predictions
valid_probs = sigmoid_np(valid_logits)
valid_labels_arr = np.array(valid_out.label_ids).astype(int)

def find_best_thresholds_multilabel(probabilities, labels, steps=99):
    n_labels = labels.shape[1]
    best_thresholds = []
    metrics = {}
    thresholds = np.linspace(0.01, 0.99, steps)
    for j in range(n_labels):
        scores = probabilities[:, j]
        truth = labels[:, j]
        if truth.sum() == 0:
            best_thresholds.append(0.5)
            metrics[j] = {'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'threshold': 0.5}
            continue
        best_f1 = -1
        best_t = 0.5
        for t in thresholds:
            preds = (scores >= t).astype(int)
            f1 = f1_score(truth, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        preds_best = (scores >= best_t).astype(int)
        p = precision_score(truth, preds_best, zero_division=0)
        r = recall_score(truth, preds_best, zero_division=0)
        best_thresholds.append(float(best_t))
        metrics[j] = {'f1': float(best_f1), 'precision': float(p), 'recall': float(r), 'threshold': float(best_t)}
    return np.array(best_thresholds), metrics

optimal_thresholds, perlabel_metrics = find_best_thresholds_multilabel(valid_probs, valid_labels_arr)
print("Optimal thresholds:", optimal_thresholds)


Optimal thresholds: [0.45 0.65 0.53 0.63 0.59 0.74 0.43 0.64 0.56 0.75 0.5 ]


In [19]:

kaggle_texts = test_df[TEXT_COL].astype(str).tolist()
kaggle_ids = test_df[ID_COL].tolist()
kaggle_ds = Dataset.from_dict({"text": kaggle_texts})
kaggle_tokenized = kaggle_ds.map(hf_tokenize, batched=True, remove_columns=["text"])
kaggle_tokenized.set_format(type="torch", columns=["input_ids","attention_mask"])

kaggle_out = trainer.predict(kaggle_tokenized)
kaggle_logits = kaggle_out.predictions
kaggle_probs = sigmoid_np(kaggle_logits)


if 'optimal_thresholds' in globals():
    th = optimal_thresholds
else:
    th = np.array([0.5]*num_labels)

kaggle_preds = (kaggle_probs >= th).astype(int)


sub_df = pd.DataFrame(kaggle_preds, columns=label_cols)
sub_df.insert(0, ID_COL, kaggle_ids)
submission_path = "submission_kaggle_binary.csv"
sub_df.to_csv(submission_path, index=False)
print("Saved Kaggle binary submission to", submission_path)


def labels_row_from_binary(row, class_names):
    positives = [name for name, val in zip(class_names, row) if val == 1]
    return " ".join(positives) if positives else ""

sub_labels = [labels_row_from_binary(row, label_cols) for row in kaggle_preds]
sub_df2 = pd.DataFrame({ID_COL: kaggle_ids, "labels": sub_labels})
submission_path2 = "submission_kaggle_labelsets.csv"
sub_df2.to_csv(submission_path2, index=False)
print("Saved Kaggle label-set submission to", submission_path2)


Map:   0%|          | 0/3259 [00:00<?, ? examples/s]

Saved Kaggle binary submission to submission_kaggle_binary.csv
Saved Kaggle label-set submission to submission_kaggle_labelsets.csv


In [20]:
meta = {
    "optimal_thresholds": optimal_thresholds.tolist() if 'optimal_thresholds' in globals() else [0.5]*num_labels,
    "label_cols": label_cols,
    "num_labels": num_labels,
    "training_args": training_args.to_dict()
}
with open("modernbert_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Saved modernbert_metadata.json")

Saved modernbert_metadata.json


In [21]:
from google.colab import files
files.download("submission_kaggle_binary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
files.download("submission_kaggle_labelsets.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ModernBert differs from the original bert in several important ways that improves the speed ,efficieny and performance on tasks.ModernBert uses a byte -level tokenizer which handles rare and noisy text compared to berts wordpiece tokenizer which often breaks words into multiple fragments.
ModernBert replaces berts absolute positional embeddings with rotary poositional embeddings ,generalizing better to longer sequences and offers more stable training.
Architecturally ModernBert includes newer optimizations such as flashattention ,improves normalization strategies and more efficient feed -forward layers,all of which reduce memory usage and significantly speed up both training and inference compared to original bert.All of these enable modernbert to train fasters ,is more stable and performs better on tasks like multi-label tweet classification,especially dealing with informal ,noisy or variable-length text.